# fitting example

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u
from astropy.io import fits

from simplefit import fit_spectrum, plot_fit

#inline matplotlib
%matplotlib inline

In [ ]:
def mean_spectrum(path):
    data = np.squeeze(fits.getdata(path))
    header = fits.getheader(path)
    if data.ndim != 3:
        raise ValueError(f"Expected a 3D cube after squeezing, got shape {data.shape}")

    spectrum = np.nanmean(data, axis=(1, 2))
    channels = np.arange(spectrum.size) + 1

    unit = u.Unit(header.get("CUNIT3", "Hz").strip() or "Hz")
    frequency = (header["CRVAL3"] + (channels - header["CRPIX3"]) * header["CDELT3"]) * unit
    rest_frequency = header.get("RESTFRQ", header.get("RESTFREQ")) * u.Hz
    velocity = ((1 - frequency.to(u.Hz) / rest_frequency) * const.c).to(u.km / u.s)
    return velocity.value, spectrum

In [ ]:
INPUT_FILE = "./example_data/HNCO_cube_20arcsec_m100_p100kms_narrowlines.fits"

## HNCO Cube Cutout And Mean Spectrum

In [ ]:
velocity, spectrum = mean_spectrum(INPUT_FILE)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(velocity, spectrum, color="black", lw=1.5)
ax.axhline(0, color="0.7", lw=1, ls="--")
ax.set_xlabel("Radio velocity (km/s)")
ax.set_ylabel("Mean intensity")
ax.set_title("HNCO mean spectrum")
ax.grid(alpha=0.25)
plt.show()

## CARTA-style Gaussian Fit


In [ ]:
fit = fit_spectrum(spectrum, velocity)

print(f"Success: {fit.success}")
print(f"Message: {fit.message}")
print(f"Noise estimate: {fit.noise:.4g}")
print()
print("Component  Center        Amplitude     FWHM         Integral")
for index, component in enumerate(fit.components, start=1):
    print(
        f"{index:>9d}  "
        f"{component.center:>10.4f}  "
        f"{component.amplitude:>11.4g}  "
        f"{component.fwhm:>10.4f}  "
        f"{component.integral:>11.4g}"
    )

fig, (ax, residual_ax) = plot_fit(fit)
ax.set_title("HNCO mean spectrum Gaussian fit")
ax.set_xlabel("")
residual_ax.set_xlabel("Radio velocity (km/s)")
fig.align_ylabels()
plt.show()

## scousepy Stage 2-style SAA automated guess

This mirrors the automated derivative-spectroscopy part of scousepy Stage 2 for a single spectrum. In a full scousepy run this would be an SAA spectrum; here the mean spectrum is used as the SAA-like input. `DSpec` identifies the initial Gaussian guesses controlled by `SNR` and `alpha`, then `Decomposer.fit_spectrum_with_guesses` sends those guesses to pyspeckit.


In [ ]:
SCOUSEPY_PATH = Path("/Users/abarnes/Library/CloudStorage/Dropbox/GitHub/scousepy")
if str(SCOUSEPY_PATH) not in sys.path:
    sys.path.insert(0, str(SCOUSEPY_PATH))

from scousepy.SpectralDecomposer import Decomposer
from scousepy.dspec import DSpec


def gaussian_sigma_model(axis, amplitude, center, sigma):
    return amplitude * np.exp(-0.5 * ((axis - center) / sigma) ** 2)


def scousepy_stage2_saa_guess_and_fit(x_data, x_axis, *, noise, snr=8, alpha=4):
    """Use scousepy's Stage 2 derivative-spectroscopy guesser on one SAA-like spectrum."""
    x_data = np.asarray(x_data, dtype=float)
    x_axis = np.asarray(x_axis, dtype=float)
    finite = np.isfinite(x_data) & np.isfinite(x_axis)
    x_data = x_data[finite]
    x_axis = x_axis[finite]

    order = np.argsort(x_axis)
    x_sorted = x_axis[order]
    y_sorted = x_data[order]

    dspec = DSpec(x_sorted, y_sorted, noise, SNR=snr, alpha=alpha, gradmethod="convolve")
    guesses = list(dspec.guesses)
    if len(guesses) == 0:
        raise RuntimeError("scousepy DSpec did not identify any positive components.")

    decomposer = Decomposer(x_sorted, y_sorted, noise)
    decomposer.create_a_spectrum()
    decomposer.fit_spectrum_with_guesses(guesses, fittype="gaussian")

    modeldict = decomposer.modeldict
    params = np.asarray(modeldict["params"], dtype=float)
    errors = np.asarray(modeldict["errors"], dtype=float)
    components = params.reshape((-1, 3))
    component_errors = errors.reshape((-1, 3))

    component_models = [gaussian_sigma_model(x_sorted, *component) for component in components]
    model = np.sum(component_models, axis=0) if component_models else np.zeros_like(y_sorted)
    residual = y_sorted - model

    guess_components = np.asarray(guesses, dtype=float).reshape((-1, 3))

    return {
        "x_axis": x_sorted,
        "x_data": y_sorted,
        "dspec": dspec,
        "guess_components": guess_components,
        "components": components,
        "component_errors": component_errors,
        "component_models": component_models,
        "model": model,
        "residual": residual,
        "modeldict": modeldict,
        "snr": snr,
        "alpha": alpha,
        "noise": noise,
    }


scouse_stage2 = scousepy_stage2_saa_guess_and_fit(spectrum, velocity, noise=fit.noise, snr=8, alpha=4)
modeldict = scouse_stage2["modeldict"]

print(f"Derivative-spectroscopy guesses: {len(scouse_stage2['guess_components'])} components")
print(f"pyspeckit fitted components: {modeldict['ncomps']}")
print(f"Fit converged: {modeldict['fitconverge']}")
print(f"AIC: {modeldict['AIC']:.3f}")
print()
print("Component  Center        Amplitude     Sigma        FWHM")
for index, (component, error) in enumerate(zip(scouse_stage2["components"], scouse_stage2["component_errors"]), start=1):
    amplitude, center, sigma = component
    print(
        f"{index:>9d}  "
        f"{center:>10.4f}  "
        f"{amplitude:>11.4g}  "
        f"{sigma:>10.4f}  "
        f"{2.354820045 * abs(sigma):>10.4f}"
    )

fig = plt.figure(figsize=(11, 7))
grid = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.25], width_ratios=[1.45, 1.0], hspace=0.28, wspace=0.28)
ax_spec = fig.add_subplot(grid[0, 0])
ax_deriv = fig.add_subplot(grid[0, 1], sharex=ax_spec)
ax_info = fig.add_subplot(grid[1, :])

x_axis = scouse_stage2["x_axis"]
y_data = scouse_stage2["x_data"]
dspec = scouse_stage2["dspec"]

ax_spec.plot(x_axis, y_data, color="#2f5f9f", lw=1.3, drawstyle="steps-mid", label="Spectrum")
ax_spec.plot(x_axis, dspec.ysmooth, color="black", lw=1.2, ls=":", label=f"Gaussian smooth alpha={scouse_stage2['alpha']}")
ax_spec.vlines(
    scouse_stage2["guess_components"][:, 1],
    0.0,
    scouse_stage2["guess_components"][:, 0],
    color="black",
    lw=1.1,
    label="DSpec guesses",
)
ax_spec.scatter(scouse_stage2["guess_components"][:, 1], scouse_stage2["guess_components"][:, 0], color="black", s=18, zorder=4)
for component_model in scouse_stage2["component_models"]:
    ax_spec.plot(x_axis, component_model, color="#29934a", lw=1.2, alpha=0.85)
ax_spec.plot(x_axis, scouse_stage2["model"], color="#c017a2", lw=2.0, label="Total model")
ax_spec.plot(x_axis, scouse_stage2["residual"], color="#df7f21", lw=1.0, drawstyle="steps-mid", label="Residual")
ax_spec.axhline(0.0, color="0.45", lw=0.8)
ax_spec.axhline(scouse_stage2["snr"] * scouse_stage2["noise"], color="black", lw=0.8, ls="--", alpha=0.55, label="SNR x rms")
ax_spec.set_title("Stage 2 derivative-spectroscopy SAA guess")
ax_spec.set_ylabel("Mean intensity")
ax_spec.legend(loc="best", frameon=False, fontsize=8)
ax_spec.grid(alpha=0.25)

for derivative, color, label in [
    (dspec.d1, "#2f5f9f", "1st derivative"),
    (dspec.d2, "#df7f21", "2nd derivative"),
    (dspec.d3, "#29934a", "3rd derivative"),
    (dspec.d4, "#d23b3b", "4th derivative"),
]:
    scale = np.nanmax(np.abs(derivative))
    if np.isfinite(scale) and scale > 0:
        derivative = derivative / scale
    ax_deriv.plot(x_axis, derivative, color=color, lw=1.0, label=label)
ax_deriv.axhline(0.0, color="0.45", lw=0.8)
ax_deriv.set_title("Derivatives of smoothed spectrum")
ax_deriv.set_ylabel("Normalized derivative")
ax_deriv.legend(loc="best", frameon=False, fontsize=8)
ax_deriv.grid(alpha=0.25)

info_lines = [
    f"Fit converged: {modeldict['fitconverge']}    n_components: {modeldict['ncomps']}    AIC: {modeldict['AIC']:.3f}    redchisq: {modeldict['redchisq']:.3g}",
    f"Derivative spectroscopy controls: SNR={scouse_stage2['snr']}    alpha={scouse_stage2['alpha']}    rms={scouse_stage2['noise']:.4g}",
    "",
    "Component   amplitude +/- error       shift +/- error          sigma +/- error          FWHM",
]
for index, (component, error) in enumerate(zip(scouse_stage2["components"], scouse_stage2["component_errors"]), start=1):
    amplitude, center, sigma = component
    amp_err, center_err, sigma_err = error
    info_lines.append(
        f"{index:>9d}   {amplitude:>9.4g} +/- {amp_err:<9.3g}   "
        f"{center:>9.4f} +/- {center_err:<9.3g}   "
        f"{sigma:>9.4f} +/- {sigma_err:<9.3g}   "
        f"{2.354820045 * abs(sigma):>9.4f}"
    )

ax_info.axis("off")
ax_info.text(0.01, 0.98, "\n".join(info_lines), ha="left", va="top", family="monospace", fontsize=9)
ax_info.set_title("pyspeckit fit information", loc="left", pad=6)

ax_spec.set_xlabel("")
ax_deriv.set_xlabel("")
ax_info.set_xlabel("Radio velocity (km/s)")
fig.align_ylabels()
plt.show()
fig